In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.cluster import DBSCAN

from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")
results_dir = Path("../results/clustering")
model_dir = Path("../models")

matrice_tfidf_bigrammes = load_npz(processed_dir / "tfidf_matrix_bigrammes.npz")
model = joblib.load(model_dir / "dbscan_champion.pkl")
labels = pd.read_csv(results_dir / "champion_labels.csv")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")


assert matrice_tfidf_bigrammes.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])
assert labels["id"].equals(ids["id"])
assert np.array_equal(labels["cluster"].to_numpy(), model.labels_)

In [3]:
clusters = ids.assign(
    cluster=model.labels_,
    products=data["description_texte"],
)

In [4]:
pd.set_option("display.max_colwidth", None)
clusters

,id,cluster,products
0,1,0,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program ."
1,2,-1,"active sport boxer briefs - skinning glory requires movement boxers deciding poach route . form - fitting active sport boxer briefs breathable 93 % polyester 71 % recycled fabric s fast - wicking , dries quickly 7 % spandex stretch seamless waistband soft leg edges won t roll bind . gusseted , flat - sewn 6 inseam size m offset prevent inner - thigh chafe . fly - free smooth panel . recyclable common threads recycling program ."
2,3,-1,"active sport briefs - superbreathable - fly briefs minimalist s choice high - octane endeavors . blend fast - wicking , quick - drying 93 % polyester 71 % recycled 7 % spandex stretch - mesh support open mesh cooling airflow . soft edging leg openings seamless waist won t roll create friction layers . smooth panel opacity . recyclable common threads recycling program ."
3,4,-1,"alpine guide pants - skin , climb ice , switch rock , traverse knife - edge ridge boogie - durable , weather - resistant breathable soft - shell pants stride mountain endeavor . midweight stretch - woven polyester won t restrict moves , brushed interior maintains - - skin comfort . 2 - way zippered fly keeps things easy harness , gusseted , zippered cuffs 2 - position snaps tie - loops . water - resistant zippered pockets , thigh , hip . recyclable common threads recycling program ."
4,5,14,"alpine wind jkt - high ridges , steep ice alpine , jacket serves true best worlds staple . excels stand - shell blustery rock climbs , cool - weather trail runs high - output ski tours . , conditions ice alpine climbing , functions lightly insulated windshirt approach , frictionless midlayer s time bundle tie - . polyester ripstop shell deluge dwr durable water repellent treatment sheds snow blocks wind , smooth , lightly brushed hanging mesh liner wicks moisture , dries fast , doesn t bind baselayers . superlight stretch - woven underarm panels enhance breathability allow unimpaired arm motion , hand pockets close zippers . drawcord hem , elastic cuffs , heat - transfer reflective logo regular - coil , center - zipper dwr finish round features . updated season improved fit . recyclable common threads recycling program ."
...,...,...,...
495,496,0,"cap 2 bottoms - cut loose maddening crowds search undone . capilene 2 fabric balances fast - drying performance lightweight insulation haven t. open - knit 4 - oz polyester 54 % recycled provides exceptional breathability , wicks perspiration , dries flash , pants ultimate comfort beneath layers chilly days . feature elastic waistband s brushed softness , seams set forward eliminate chafe . gusseted crotch , functional fly gladiodor natural odor control garment . recyclable common threads recycling program ."
496,497,0,"cap 2 crew - crew takes edge fickle weather . clearing conditions , fastest dry time performance baselayer fabrics , open - knit construction breathes wicks dry cool . weather window shuts , wear insulation beneath layers . self - fabric neck s soft skin , raglan sleeves flat - ribbed underarm panels merge way shoulder straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
497,498,1,"- time shell - need use morning times umbrella . - times handsome matte - finish , 100 % recycled polyester shell provides tasteful sports - coat - length coverage vicissitudes weather city country . hardy 2.5 - layer , 4.2 - oz polyester ripstop h2no waterproof / breathable barrier deluge dwr durable water repellent finish scoffs cloudbursts 100

In [7]:
def item_name_to_id(saisie):
    """
    Cette fonction prend un nom de produit en entrée et renvoie l'identifiant du produit correspondant.
    """
    # Vérifier si le nom du produit est présent dans le DataFrame clusters
    if saisie not in clusters["products"].values:
        raise ValueError(f"Le nom du produit '{saisie}' n'existe pas dans le DataFrame clusters.")
    
    item_id = clusters.loc[clusters["products"] == saisie, "id"].values[0]
    
    return item_id

In [ ]:
def recommander(saisie):
    
    while True:
        saisie = saisie.strip()

        try:
            item_id = int(saisie) if saisie.isdigit() else item_name_to_id(saisie)
            input_item = clusters.loc[clusters["id"] == item_id]
        except ValueError as erreur:
            print(erreur)
            saisie = input("Entrez un nom ou un identifiant produit : ")
            continue

        if input_item.empty:
            print("Produit introuvable. Veuillez réessayer.")
        elif input_item["cluster"].iloc[0] == -1:
            print("Ce produit appartient au bruit. Veuillez choisir un autre produit.")
        else:
            break

        saisie = input("Entrez un nom ou un identifiant produit : ")

    cluster = input_item["cluster"].iloc[0]
    cluster_items = clusters.loc[clusters["cluster"] == cluster].copy()

    cluster_items["score"] = cosine_similarity(
        matrice_tfidf_bigrammes[[input_item.index[0]]],
        matrice_tfidf_bigrammes[cluster_items.index.to_numpy()],
    ).ravel()

    cluster_items.sort_values(by="score", ascending=False, inplace=True)
    recommandation = cluster_items.drop(index=input_item.index).head(5)
    
    return recommandation

In [35]:
saisie = input("Entrez un nom ou un identifiant produit : ")
recommander(saisie)

,id,cluster,products,score
499,500,5,"- wear shorts - time simplify - wear shorts prove short . comfortable garment - washed 100 % organic cotton canvas , withstand hard work arduous journeys remain unfazed . durable soft touch , feature wide - entry drop - pockets , rear welted pockets 8 1/2 inseam size 32 . recyclable common threads recycling program .",0.561432
462,463,5,"custodian pants - short - graveyard shift advantages quitting time dawn patrol . custodian pants defined strong work ethic . tough comfortable 5.2 - oz organic cotton canvas , flat , generous wide - entry pockets , welted pockets , zip fly button closure . inseams size 32 short , 30 regular , 32 long , 34 . recyclable common threads recycling program .",0.546339
461,462,5,"custodian pants - reg - graveyard shift advantages quitting time dawn patrol . custodian pants defined strong work ethic . tough comfortable 5.2 - oz organic cotton canvas , flat , generous wide - entry pockets , welted pockets , zip fly button closure . inseams size 32 short , 30 regular , 32 long , 34 . recyclable common threads recycling program .",0.541140
31,32,5,"custodian pants - long - graveyard shift advantages quitting time dawn patrol . custodian pants defined strong work ethic . tough comfortable 5.2 - oz organic cotton canvas , flat , generous wide - entry pockets , welted pockets , zip fly button closure . inseams size 32 short , 30 regular , 32 long , 34 . recyclable common threads recycling program .",0.541113
33,34,5,"delivery shorts - locals know best spots . delivery shorts built - day comfort durability , simple , practical style . blend 65 % - recycled polyester 35 % organic cotton , designed nod local deliverymen . features include flat offset seams zip fly button closure , generous wide - entry pockets welted pockets . inseam size 32 11 . recyclable common threads recycling program .",0.514458
